# 01. ReAct Prototype V1

BA-Agent-Bench의 동일한 입력 문서로 파이프라인을 비교하기 위한 단일 ReAct 기준선입니다.

- Agent에는 `golden_stories`를 제공하지 않습니다.
- Tool은 현재 case의 문서에만 접근합니다.
- 정확도, latency, token과 citation 지표는 Agent 밖에서 기록합니다.
- 이 Notebook의 lexical 정확도와 citation 지표는 smoke metric이며 최종 hallucination 판정이 아닙니다.

## 0. 실행 전 준비

프로젝트 Poetry 환경에 `langchain`, `langgraph`, `langchain-openai`, `python-dotenv`가 설치되어 있어야 합니다. API key와 모델은 `.env` 또는 실행 환경에서 설정합니다.

```text
OPENAI_API_KEY=
OPENAI_MODEL=gpt-5-mini
```

모델 비교 시에는 코드가 아니라 `OPENAI_MODEL`만 변경합니다.

In [ ]:
from collections import Counter
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter
from typing import Annotated, Any
from urllib.parse import urlencode
from urllib.request import Request, urlopen
import json
import math
import operator
import os
import re

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import MessagesState

load_dotenv()

DATASET_REPO = "CentificAIResearch/BA-Agent-Bench"
DATASET_CONFIG = "default"
DATASET_SPLIT = "train"
PIPELINE_ID = "react_v1"
MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-5-mini")
MAX_TOOL_RESULTS = 5
MAX_AGENT_STEPS = 30

if not os.getenv("OPENAI_API_KEY"):
    print("주의: OPENAI_API_KEY가 없어 Agent 실행 셀은 실패합니다.")

print({"pipeline": PIPELINE_ID, "model": MODEL_NAME, "dataset": DATASET_REPO})

## 1. Hugging Face 평가 데이터 로드

Dataset Viewer API를 사용하므로 별도의 `datasets` 패키지가 필요하지 않습니다. 공개 subset은 8개 case로 구성됩니다.

In [ ]:
def load_huggingface_rows(dataset_repo: str, config: str, split: str, limit: int = 100) -> list[dict[str, Any]]:
    query = urlencode({
        "dataset": dataset_repo,
        "config": config,
        "split": split,
        "offset": 0,
        "length": limit
    })
    request = Request(
        f"https://datasets-server.huggingface.co/rows?{query}",
        headers={"User-Agent": "freelance-ops-agent-evaluation/1.0"}
    )
    with urlopen(request, timeout=30) as response:
        payload = json.load(response)
    return [item["row"] for item in payload["rows"]]


cases = load_huggingface_rows(
    dataset_repo=DATASET_REPO,
    config=DATASET_CONFIG,
    split=DATASET_SPLIT
)

assert cases, "Hugging Face dataset이 비어 있습니다."
assert len({case["task_id"] for case in cases}) == len(cases), "task_id가 중복되었습니다."

print(f"case_count={len(cases)}")
print([(case["task_id"], len(case["input_documents"]), len(case["golden_stories"])) for case in cases])

## 2. Case별 문서 Workspace

Agent에게 `task_id` 선택 권한을 주지 않습니다. 평가 하네스가 현재 case를 Workspace에 바인딩하여 다른 test case나 Golden Story에 접근하지 못하게 합니다.

In [ ]:
@dataclass(frozen=True, slots=True)
class DocumentChunk:
    document_id: str
    document_title: str
    chunk_id: str
    content: str


def tokenize(text: str) -> set[str]:
    return {
        token.lower()
        for token in re.findall(r"[A-Za-z0-9가-힣_-]{2,}", text)
    }


def split_content(content: str, max_characters: int = 1400) -> list[str]:
    paragraphs = [paragraph.strip() for paragraph in re.split(r"\n\s*\n", content) if paragraph.strip()]
    chunks = []
    buffer = ""
    for paragraph in paragraphs:
        candidate = f"{buffer}\n\n{paragraph}".strip()
        if buffer and len(candidate) > max_characters:
            chunks.append(buffer)
            buffer = paragraph
        else:
            buffer = candidate
    if buffer:
        chunks.append(buffer)
    return chunks or [content]


class CaseDocumentWorkspace:
    def __init__(self, case: dict[str, Any]):
        self.task_id = case["task_id"]
        documents = [{
            "doc_id": "feature-description",
            "title": case["title"],
            "content": case["description"]
        }, *case["input_documents"]]
        self.documents = []
        self.chunks = {}
        for document_index, document in enumerate(documents, start=1):
            document_id = str(document.get("doc_id") or f"document-{document_index}")
            title = str(document.get("title") or document.get("filename") or document_id)
            content = str(document.get("content") or "")
            self.documents.append({
                "document_id": document_id,
                "title": title,
                "chunk_count": len(split_content(content))
            })
            for chunk_index, chunk_content in enumerate(split_content(content), start=1):
                chunk_id = f"{document_id}::chunk-{chunk_index:03d}"
                self.chunks[chunk_id] = DocumentChunk(
                    document_id=document_id,
                    document_title=title,
                    chunk_id=chunk_id,
                    content=chunk_content
                )

    def list_documents(self) -> list[dict[str, Any]]:
        return self.documents

    def search(self, query: str, top_k: int) -> list[dict[str, Any]]:
        query_tokens = tokenize(query)
        scored = []
        for chunk in self.chunks.values():
            chunk_tokens = tokenize(chunk.content)
            overlap = len(query_tokens & chunk_tokens)
            score = overlap / math.sqrt(max(len(query_tokens), 1) * max(len(chunk_tokens), 1))
            if score > 0:
                scored.append((score, chunk))
        scored.sort(key=lambda item: (-item[0], item[1].chunk_id))
        return [
            {
                "document_id": chunk.document_id,
                "document_title": chunk.document_title,
                "chunk_id": chunk.chunk_id,
                "content": chunk.content,
                "score": round(score, 6)
            }
            for score, chunk in scored[:top_k]
        ]

    def read(self, chunk_ids: list[str]) -> list[dict[str, Any]]:
        return [asdict(self.chunks[chunk_id]) for chunk_id in chunk_ids if chunk_id in self.chunks]

    def valid_chunk_ids(self) -> set[str]:
        return set(self.chunks)


## 3. ReAct Tool

`validate_deliverable`은 형식과 인용 ID만 검사합니다. Golden Story와 비교하지 않으므로 정답 누수가 없습니다.

In [ ]:
def extract_json_object(text: str) -> dict[str, Any]:
    stripped = text.strip()
    if stripped.startswith("```"):
        stripped = re.sub(r"^```(?:json)?\s*", "", stripped)
        stripped = re.sub(r"\s*```$", "", stripped)
    start = stripped.find("{")
    end = stripped.rfind("}")
    if start < 0 or end < start:
        raise ValueError("JSON object를 찾을 수 없습니다.")
    return json.loads(stripped[start:end + 1])


def validate_payload(payload: dict[str, Any], valid_chunk_ids: set[str]) -> dict[str, Any]:
    errors = []
    stories = payload.get("user_stories")
    if not isinstance(stories, list) or not stories:
        return {"valid": False, "errors": ["user_stories는 비어 있지 않은 배열이어야 합니다."]}

    seen_titles = set()
    for index, story in enumerate(stories, start=1):
        title = str(story.get("title") or "").strip()
        description = str(story.get("description") or "").strip()
        acceptance_criteria = story.get("acceptance_criteria")
        evidence = story.get("evidence")
        if not title:
            errors.append(f"story[{index}].title 누락")
        elif title.lower() in seen_titles:
            errors.append(f"story[{index}].title 중복")
        seen_titles.add(title.lower())
        if not description:
            errors.append(f"story[{index}].description 누락")
        if not isinstance(acceptance_criteria, list) or not acceptance_criteria:
            errors.append(f"story[{index}].acceptance_criteria 누락")
        if not isinstance(evidence, list) or not evidence:
            errors.append(f"story[{index}].evidence 누락")
        else:
            for citation in evidence:
                chunk_id = citation.get("chunk_id") if isinstance(citation, dict) else None
                if chunk_id not in valid_chunk_ids:
                    errors.append(f"story[{index}]의 존재하지 않는 chunk_id: {chunk_id}")
    return {"valid": not errors, "errors": errors}


def build_tools(workspace: CaseDocumentWorkspace) -> list[Any]:
    @tool
    def list_documents() -> str:
        """현재 요구사항 case에서 읽을 수 있는 문서 목록과 chunk 수를 반환한다."""
        return json.dumps(workspace.list_documents(), ensure_ascii=False)

    @tool
    def search_documents(query: str, top_k: int = MAX_TOOL_RESULTS) -> str:
        """현재 case 문서에서 query와 관련된 원문 chunk를 결정적으로 검색한다."""
        safe_top_k = min(max(top_k, 1), MAX_TOOL_RESULTS)
        return json.dumps(workspace.search(query, safe_top_k), ensure_ascii=False)

    @tool
    def read_document_chunks(chunk_ids: list[str]) -> str:
        """검색으로 확인한 chunk_id 목록의 원문을 반환한다. 존재하지 않는 ID는 제외한다."""
        return json.dumps(workspace.read(chunk_ids), ensure_ascii=False)

    @tool
    def validate_deliverable(payload_json: str) -> str:
        """최종 user story JSON의 필수 필드와 evidence chunk_id 유효성을 검사한다."""
        try:
            payload = extract_json_object(payload_json)
            result = validate_payload(payload, workspace.valid_chunk_ids())
        except (TypeError, ValueError, json.JSONDecodeError) as error:
            result = {"valid": False, "errors": [str(error)]}
        return json.dumps(result, ensure_ascii=False)

    return [list_documents, search_documents, read_document_chunks, validate_deliverable]


SYSTEM_PROMPT = """
You are a single ReAct business-analysis agent.
Your job is to turn the supplied enterprise feature into a rigorous set of user stories.

Rules:
1. Inspect the available documents with tools. Do not invent document or chunk IDs.
2. Search and read enough source chunks before drafting stories.
3. Decompose by actor, workflow, exception, validation, and operational concern when supported.
4. Every story must contain title, description, acceptance_criteria, story_points, evidence, and assumptions.
5. Evidence is a list of objects containing document_id and chunk_id.
6. Unsupported ideas must be explicit assumptions, not facts.
7. Call validate_deliverable with the complete candidate JSON before answering.
8. If validation fails, correct the candidate and validate it again.
9. Return only one JSON object. Do not expose hidden reasoning or chain-of-thought.

Output schema:
{
  "user_stories": [
    {
      "title": "string",
      "description": "As a..., I want..., so that...",
      "acceptance_criteria": ["string"],
      "story_points": 1,
      "evidence": [{"document_id": "string", "chunk_id": "string"}],
      "assumptions": ["string"]
    }
  ],
  "unresolved_questions": ["string"]
}
""".strip()

## 4. 실행 기록과 smoke metric

Golden Story 제목과의 lexical 유사도는 빠른 회귀 확인용입니다. 최종 정확도는 benchmark rubric 또는 별도 LLM Judge와 사람 평가를 추가해야 합니다.

In [ ]:
def message_text(message: AIMessage) -> str:
    if isinstance(message.content, str):
        return message.content
    if isinstance(message.content, list):
        return "\n".join(
            str(item.get("text") or "") if isinstance(item, dict) else str(item)
            for item in message.content
        )
    return str(message.content)


def collect_usage(messages: list[Any]) -> dict[str, int]:
    input_tokens = 0
    output_tokens = 0
    for message in messages:
        usage = getattr(message, "usage_metadata", None) or {}
        input_tokens += int(usage.get("input_tokens") or 0)
        output_tokens += int(usage.get("output_tokens") or 0)
    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": input_tokens + output_tokens
    }


def lexical_story_score(predicted: list[dict[str, Any]], golden: list[dict[str, Any]]) -> float:
    predicted_titles = [tokenize(str(story.get("title") or "")) for story in predicted]
    if not golden:
        return float(not predicted)
    scores = []
    for golden_story in golden:
        golden_tokens = tokenize(str(golden_story.get("title") or ""))
        similarities = []
        for predicted_tokens in predicted_titles:
            union = golden_tokens | predicted_tokens
            similarities.append(len(golden_tokens & predicted_tokens) / len(union) if union else 0.0)
        scores.append(max(similarities, default=0.0))
    return sum(scores) / len(scores)


def citation_metrics(stories: list[dict[str, Any]], valid_chunk_ids: set[str]) -> dict[str, float]:
    citations = [
        citation
        for story in stories
        for citation in story.get("evidence", [])
        if isinstance(citation, dict)
    ]
    valid_citations = sum(citation.get("chunk_id") in valid_chunk_ids for citation in citations)
    supported_stories = sum(bool(story.get("evidence")) for story in stories)
    return {
        "citation_validity_rate": valid_citations / len(citations) if citations else 0.0,
        "stories_with_evidence_rate": supported_stories / len(stories) if stories else 0.0
    }


def evaluate_smoke(case: dict[str, Any], payload: dict[str, Any], workspace: CaseDocumentWorkspace) -> dict[str, Any]:
    predicted_stories = payload.get("user_stories", [])
    golden_stories = case["golden_stories"]
    acceptance_criteria_present = sum(bool(story.get("acceptance_criteria")) for story in predicted_stories)
    return {
        "predicted_story_count": len(predicted_stories),
        "golden_story_count": len(golden_stories),
        "story_count_absolute_error": abs(len(predicted_stories) - len(golden_stories)),
        "gold_title_lexical_score": lexical_story_score(predicted_stories, golden_stories),
        "acceptance_criteria_presence_rate": acceptance_criteria_present / len(predicted_stories) if predicted_stories else 0.0,
        **citation_metrics(predicted_stories, workspace.valid_chunk_ids())
    }


def run_react_case(case: dict[str, Any]) -> dict[str, Any]:
    workspace = CaseDocumentWorkspace(case)
    tools = build_tools(workspace)
    model = ChatOpenAI(
        model=MODEL_NAME,
        max_completion_tokens=8192
    )
    agent = create_agent(
        model=model,
        tools=tools,
        system_prompt=SYSTEM_PROMPT
    )
    user_message = (
        f"Task ID: {case['task_id']}\n"
        f"Feature title: {case['title']}\n"
        f"Feature description: {case['description']}\n\n"
        "Use the bound document workspace to produce the requested deliverable."
    )
    started_at = datetime.now(timezone.utc).isoformat()
    start = perf_counter()
    try:
        result = agent.invoke(
            {"messages": [{"role": "user", "content": user_message}]},
            config={"recursion_limit": MAX_AGENT_STEPS}
        )
        latency_ms = round((perf_counter() - start) * 1000, 3)
        messages = result["messages"]
        final_message = next(message for message in reversed(messages) if isinstance(message, AIMessage))
        final_text = message_text(final_message)
        payload = extract_json_object(final_text)
        validation = validate_payload(payload, workspace.valid_chunk_ids())
        tool_trace = [
            {"name": message.name, "status": getattr(message, "status", "success")}
            for message in messages
            if isinstance(message, ToolMessage)
        ]
        return {
            "pipeline_id": PIPELINE_ID,
            "dataset_repo": DATASET_REPO,
            "task_id": case["task_id"],
            "model": MODEL_NAME,
            "started_at": started_at,
            "latency_ms": latency_ms,
            **collect_usage(messages),
            "tool_trace": tool_trace,
            "tool_call_count": len(tool_trace),
            "validation": validation,
            "smoke_metrics": evaluate_smoke(case, payload, workspace),
            "prediction": payload,
            "success": validation["valid"],
            "error": None
        }
    except Exception as error:
        return {
            "pipeline_id": PIPELINE_ID,
            "dataset_repo": DATASET_REPO,
            "task_id": case["task_id"],
            "model": MODEL_NAME,
            "started_at": started_at,
            "latency_ms": round((perf_counter() - start) * 1000, 3),
            "success": False,
            "error": f"{type(error).__name__}: {error}"
        }

## 5. 단일 case Smoke Test

먼저 한 건만 실행하여 Tool trace, JSON parsing과 비용 발생 범위를 확인합니다.

In [ ]:
smoke_result = run_react_case(cases[0])
print(json.dumps({
    "task_id": smoke_result["task_id"],
    "success": smoke_result["success"],
    "latency_ms": smoke_result["latency_ms"],
    "total_tokens": smoke_result.get("total_tokens"),
    "tool_trace": smoke_result.get("tool_trace"),
    "validation": smoke_result.get("validation"),
    "smoke_metrics": smoke_result.get("smoke_metrics"),
    "error": smoke_result.get("error")
}, ensure_ascii=False, indent=2))

## 6. 전체 dataset 실행

Smoke Test가 성공한 뒤에만 실행합니다. 첫 실행은 반복 1회로 비용을 확인하고, 최종 비교에서 동일 모델·동일 반복 횟수를 세 파이프라인에 적용합니다.

In [ ]:
RUN_FULL_DATASET = False
REPETITIONS = 1
react_results = []

if RUN_FULL_DATASET:
    for repetition in range(1, REPETITIONS + 1):
        for case in cases:
            result = run_react_case(case)
            result["repetition"] = repetition
            react_results.append(result)
            print(case["task_id"], repetition, result["success"], result["latency_ms"])
else:
    print("RUN_FULL_DATASET=False: 전체 실행을 건너뜁니다.")

## 7. 결과 저장

원본 응답에는 공개 데이터셋 내용만 포함되어야 합니다. API key와 환경변수는 저장하지 않습니다.

In [ ]:
SAVE_RESULTS = False

if SAVE_RESULTS and react_results:
    repository_root = Path.cwd()
    if not (repository_root / "test").exists():
        repository_root = repository_root.parent.parent
    result_directory = repository_root / "test" / "results" / "local"
    result_directory.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    result_path = result_directory / f"{PIPELINE_ID}_{MODEL_NAME}_{timestamp}.jsonl"
    with result_path.open("w", encoding="utf-8") as result_file:
        for result in react_results:
            result_file.write(json.dumps(result, ensure_ascii=False) + "\n")
    print(result_path)
else:
    print("저장을 건너뜁니다.")